# Activation Steering: Causal Test of Probe Directions

**Goal**: Test whether the linear probe directions found in Phase 1 are *causal* — does adding the direction to the residual stream during generation actually change whether the model follows system vs user instructions?

**Method**: For each Condition C sample, generate with `alpha * direction` added at the best probe layer. Sweep alpha from negative (push toward user) to positive (push toward system) and measure SCR.

**Directions tested**: Probe weight vector (supervised) and CMD (class-mean difference, unsupervised).

**Modules**: `steer.py` (steering helpers), `data.py` (config, prompts), `probe.py` (probe results)

In [ ]:
import sys
from pathlib import Path

# Ensure phase1_linear_probing/ is on the import path regardless of kernel cwd
_PHASE1_DIR = Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd()
if _PHASE1_DIR.name != "phase1_linear_probing":
    _PHASE1_DIR = _PHASE1_DIR / "phase1_linear_probing"
sys.path.insert(0, str(_PHASE1_DIR))

import importlib
import json
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer
import threadpoolctl
threadpoolctl.threadpool_limits(2, "blas")

import data as _data_mod, probe as _probe_mod, steer as _steer_mod
for _m in [_data_mod, _probe_mod, _steer_mod]:
    importlib.reload(_m)

from data import (
    ProbeConfig, find_repo_root, load_sync_env,
    load_results, prepare_condition_c,
    build_formatted_prompt,
    _load_nn_model, _cleanup_nn_model,
)
from probe import (
    load_results as load_probe_results, results_path, ProbeResult,
)
from steer import (
    load_steering_directions, steer_and_generate,
    score_steered_output, run_steering_sweep, compute_steered_scr,
)

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
PROBE_CONFLICTS_8B = [
    "alliteration_density",
    "capitalization_all_caps",
    "direct_answer_vs_hedging",
    "emoji_use_vs_avoid",
    "formal_vs_casual_tone",
    "format_json_markdown",
    "html_emphasis_tags",
    "imperative_vs_declarative",
    "json_only_vs_plain",
    "language_en_es",
    "language_en_zh",
    "leetspeak_encoding",
    "list_bullets_vs_numbered",
    "lowercase_vs_capitalized",
    "numbered_sections_vs_prose",
    "parenthetical_asides",
    "pronoun_density",
    "questions_vs_statements",
    "sentence_connector_density",
    "short_paragraphs_vs_single_block",
    "short_vs_long_sentences",
    "spanish_loanwords",
    "starting_word_hello_greetings",
    "vowel_omission",
]

cfg = ProbeConfig(
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    label_mode="binary",
    token_positions=["last_prompt"],
    cv_mode="grouped",
    n_cv_folds=24,
    use_scaler=False,
    probe_C=1.0,
    batch_size=1,
    conflict_ids=PROBE_CONFLICTS_8B,
    run_id="curated24-8b-v002",
)
load_sync_env(cfg.repo_root)
cfg.ensure_dirs()

# Steering parameters
N_PER_CONFLICT = 2          # samples per conflict for sweep (~120 total)
ALPHAS = [3, 6, 10]
MAX_NEW_TOKENS = 256

print(f"Run ID       : {cfg.run_id}")
print(f"Run dir      : {cfg.run_dir}")
print(f"Device       : {cfg.device}")
print(f"Model        : {cfg.model_name}")
print(f"Alphas       : {ALPHAS}")
print(f"N/conflict   : {N_PER_CONFLICT}")

In [ ]:
# ── Load probe results, find best layer ───────────────────────────────────────
pos = cfg.token_positions[0]
rpath = results_path(cfg.run_dir, cfg.cv_mode, cfg.use_scaler)
results = load_probe_results(rpath)
pr = results[pos]

best_layer = int(pr.cv_scores.loc[pr.cv_scores["roc_auc_mean"].idxmax(), "layer"])
best_auc = pr.cv_scores.loc[pr.cv_scores["roc_auc_mean"].idxmax(), "roc_auc_mean"]
print(f"Best layer   : {best_layer}")
print(f"Best AUC     : {best_auc:.3f}")

In [ ]:
# ── Load steering directions, print cosine similarity ─────────────────────────
directions = load_steering_directions(cfg.run_dir, pos, best_layer)

print(f"Directions loaded: {list(directions.keys())}")
print(f"Probe shape      : {directions['probe'].shape}")

if "cmd_overall" in directions:
    cos_sim = np.dot(directions["probe"], directions["cmd_overall"])
    print(f"Probe–CMD cosine : {cos_sim:.4f}")

if "cmd_per_constraint" in directions:
    cpc = directions["cmd_per_constraint"]
    sims = {k: np.dot(directions["probe"], v) for k, v in cpc.items()}
    print(f"\nProbe–constraint CMD cosine similarities:")
    for k, s in sorted(sims.items(), key=lambda x: -x[1]):
        print(f"  {k:<40} {s:+.4f}")

In [ ]:
# ── Load Condition C samples, subsample ───────────────────────────────────────
df_all = load_results(cfg.data_dir, cfg.model_name)
df_c = prepare_condition_c(df_all, cfg.label_mode, conflict_ids=cfg.conflict_ids)

# Subsample N_PER_CONFLICT per conflict
df_samples = (
    df_c.groupby("conflict_id", group_keys=False)
    .sample(n=N_PER_CONFLICT, random_state=42)
    .reset_index(drop=True)
)

n_conflicts = df_samples["conflict_id"].nunique()
phase0_scr = (df_c["label"] == "followed_system").mean()

print(f"Full Condition C : {len(df_c)} samples, SCR={phase0_scr:.3f}")
print(f"Subsampled       : {len(df_samples)} samples across {n_conflicts} conflicts")
print(f"Sweep size       : {len(df_samples)} × {len(ALPHAS)} alphas = {len(df_samples) * len(ALPHAS)} generations")

In [ ]:
# ── Load model ────────────────────────────────────────────────────────────────
model_nn, n_layers = _load_nn_model(cfg)
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
print(f"Model loaded: {n_layers} layers")

In [ ]:
# ── Smoke test: alpha=0, single sample ────────────────────────────────────────
test_row = df_samples.iloc[0]
test_prompt = build_formatted_prompt(
    tokenizer, test_row["system_prompt"], test_row["user_prompt"]
)

print(f"Conflict: {test_row['conflict_id']}")
print(f"Direction: {test_row['direction']}")
print(f"System: {test_row['system_prompt'][:80]}...")
print(f"User: {test_row['user_prompt'][:80]}...")
print(f"Prompt tokens: {len(tokenizer.encode(test_prompt))}")
print()

response = steer_and_generate(
    model_nn, tokenizer, test_prompt,
    directions["probe"], best_layer, alpha=0.0,
    max_new_tokens=MAX_NEW_TOKENS,
)
print(f"Response (alpha=0):\n{response[:500]}")
print(f"\n--- Response length: {len(response)} chars ---")

In [ ]:
# ── Baseline: alpha=0 sweep, compare SCR to Phase 0 ──────────────────────────
sweep_baseline = run_steering_sweep(
    model_nn, tokenizer, df_samples,
    directions["probe"], best_layer,
    alphas=[0.0],
    max_new_tokens=MAX_NEW_TOKENS,
)

baseline_scr = compute_steered_scr(sweep_baseline, alpha=0.0)
print(f"Phase 0 SCR (full data) : {phase0_scr:.3f}")
print(f"Baseline SCR (alpha=0)  : {baseline_scr:.3f}")
print(f"Delta                   : {baseline_scr - phase0_scr:+.3f}")
print(f"\nLabel distribution at alpha=0:")
print(sweep_baseline["label"].value_counts().to_string())

In [ ]:
# ── Probe direction sweep ─────────────────────────────────────────────────────
sweep_probe = run_steering_sweep(
    model_nn, tokenizer, df_samples,
    directions["probe"], best_layer,
    alphas=ALPHAS,
    max_new_tokens=MAX_NEW_TOKENS,
)

scr_probe = compute_steered_scr(sweep_probe)
print("Probe direction SCR by alpha:")
print(scr_probe.to_string())

In [ ]:
# ── CMD direction sweep ───────────────────────────────────────────────────────
if "cmd_overall" in directions:
    sweep_cmd = run_steering_sweep(
        model_nn, tokenizer, df_samples,
        directions["cmd_overall"], best_layer,
        alphas=ALPHAS,
        max_new_tokens=MAX_NEW_TOKENS,
    )

    scr_cmd = compute_steered_scr(sweep_cmd)
    print("CMD direction SCR by alpha:")
    print(scr_cmd.to_string())
else:
    sweep_cmd = None
    scr_cmd = None
    print("CMD direction not available — skipping")

In [ ]:
# ── Save sweep results ────────────────────────────────────────────────────────
steer_dir = cfg.run_dir / "steering"
steer_dir.mkdir(exist_ok=True)

sweep_probe.to_json(steer_dir / "sweep_probe.jsonl", orient="records", lines=True)
print(f"Saved: {steer_dir / 'sweep_probe.jsonl'} ({len(sweep_probe)} rows)")

if sweep_cmd is not None:
    sweep_cmd.to_json(steer_dir / "sweep_cmd.jsonl", orient="records", lines=True)
    print(f"Saved: {steer_dir / 'sweep_cmd.jsonl'} ({len(sweep_cmd)} rows)")

sweep_baseline.to_json(steer_dir / "sweep_baseline.jsonl", orient="records", lines=True)
print(f"Saved: {steer_dir / 'sweep_baseline.jsonl'} ({len(sweep_baseline)} rows)")

In [ ]:
# ── Cleanup model ─────────────────────────────────────────────────────────────
_cleanup_nn_model(model_nn, cfg.device)
del model_nn
print("Model cleaned up")

## Visualization

Load results from disk if needed (no GPU required for plotting).

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── SCR vs Alpha curves ──────────────────────────────────────────────────────
fig = go.Figure()

# Probe direction
fig.add_trace(go.Scatter(
    x=scr_probe.index.tolist(), y=scr_probe.values.tolist(),
    mode="lines+markers", name="Probe direction",
    line=dict(color="blue", width=2),
    marker=dict(size=8),
))

# CMD direction
if scr_cmd is not None:
    fig.add_trace(go.Scatter(
        x=scr_cmd.index.tolist(), y=scr_cmd.values.tolist(),
        mode="lines+markers", name="CMD direction",
        line=dict(color="red", width=2, dash="dash"),
        marker=dict(size=8),
    ))

# Phase 0 baseline
fig.add_hline(
    y=phase0_scr, line_dash="dot", line_color="gray",
    annotation_text=f"Phase 0 SCR = {phase0_scr:.3f}",
)

fig.update_layout(
    title=f"Activation Steering: SCR vs Alpha (Layer {best_layer})",
    xaxis_title="Alpha (steering strength)",
    yaxis_title="System Compliance Rate (SCR)",
    yaxis=dict(range=[0, 1]),
    template="plotly_white",
    width=800, height=500,
    legend=dict(x=0.02, y=0.98),
)
fig.show()

In [ ]:
# ── Per-conflict SCR heatmap ──────────────────────────────────────────────────
# Rows = conflict_id, columns = alpha, values = SCR
pivot = sweep_probe.copy()
pivot["sys"] = (pivot["label"] == "followed_system").astype(int)
heatmap_data = pivot.pivot_table(
    index="conflict_id", columns="alpha", values="sys", aggfunc="mean"
)
heatmap_data = heatmap_data.reindex(columns=sorted(heatmap_data.columns))

fig = go.Figure(data=go.Heatmap(
    z=heatmap_data.values,
    x=[str(a) for a in heatmap_data.columns],
    y=heatmap_data.index.tolist(),
    colorscale="RdBu",
    zmid=0.5,
    zmin=0, zmax=1,
    text=np.round(heatmap_data.values, 2).astype(str),
    texttemplate="%{text}",
    colorbar=dict(title="SCR"),
))

fig.update_layout(
    title=f"Per-Conflict SCR by Alpha (Probe Direction, Layer {best_layer})",
    xaxis_title="Alpha",
    yaxis_title="Conflict ID",
    template="plotly_white",
    height=max(400, len(heatmap_data) * 25 + 100),
    width=900,
)
fig.show()

In [ ]:
# ── Label distribution stacked bars ───────────────────────────────────────────
label_order = ["followed_system", "followed_user", "followed_both", "followed_neither"]
colors = {"followed_system": "#2166ac", "followed_user": "#b2182b",
          "followed_both": "#92c5de", "followed_neither": "#f4a582"}

fig = go.Figure()

for label in label_order:
    counts = []
    for alpha in sorted(sweep_probe["alpha"].unique()):
        sub = sweep_probe[sweep_probe["alpha"] == alpha]
        counts.append((sub["label"] == label).sum() / len(sub))
    fig.add_trace(go.Bar(
        name=label,
        x=[str(a) for a in sorted(sweep_probe["alpha"].unique())],
        y=counts,
        marker_color=colors.get(label, "gray"),
    ))

fig.update_layout(
    barmode="stack",
    title=f"Label Distribution by Alpha (Probe Direction, Layer {best_layer})",
    xaxis_title="Alpha",
    yaxis_title="Fraction",
    yaxis=dict(range=[0, 1]),
    template="plotly_white",
    width=800, height=450,
    legend=dict(x=0.02, y=0.98),
)
fig.show()

In [ ]:
# ── Qualitative examples ──────────────────────────────────────────────────────
# Show a few examples at different alphas for one conflict
example_conflict = df_samples["conflict_id"].value_counts().index[0]
example_alphas = [-5, 0, 5, 15]

print(f"=== Qualitative Examples: {example_conflict} ===\n")

example_rows = sweep_probe[
    (sweep_probe["conflict_id"] == example_conflict)
    & (sweep_probe["alpha"].isin(example_alphas))
].sort_values(["alpha"])

# Show system/user prompts once
first = example_rows.iloc[0]
print(f"System: {first['system_prompt'][:120]}")
print(f"User:   {first['user_prompt'][:120]}")
print()

for _, row in example_rows.iterrows():
    print(f"--- alpha={row['alpha']:+.0f} | label={row['label']} ---")
    print(row["response"][:300])
    print()